# Segmentation EDA: Pothole-Mix and Norwegian Road Damage

Goal: understand the data well enough to choose **input resolution**, **loss**, **sampling** and to describe the **domain gap**
between the training data (Pothole-Mix) and the evaluation data (all 50 NRD images, zero-shot, binary damage vs background).

Figures are saved to `segmentation/results/eda/` for the presentation.

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common/paths.py").exists())
sys.path.insert(0, str(REPO))
from common.paths import POTHOLE_MIX, NRD
from segmentation.dataset import PotholeMix, NorwegianRoadDamage, decode_pothole_mix_mask, CRACK, POTHOLE

FIG = REPO / "segmentation/results/eda"
FIG.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)
print(POTHOLE_MIX, NRD, sep="\n")

## 1. Size of each split and subset

In [ ]:
splits = {s: PotholeMix(s) for s in ("training", "validation", "testing")}
nrd = NorwegianRoadDamage()
counts = pd.DataFrame({s: Counter(sub for *_, sub in ds.items) for s, ds in splits.items()}).fillna(0).astype(int)
counts.loc["TOTAL"] = counts.sum()
counts["share_of_train_%"] = (100 * counts["training"] / counts.loc["TOTAL", "training"]).round(1)
print("NRD images:", len(nrd))
counts

## 2. Per-image statistics (masks only)

Reads every mask once; cached to `data/eda_pothole_mix_stats.csv` (gitignored) because it takes a minute or two.

In [ ]:
CACHE = REPO / "data/eda_pothole_mix_stats.csv"
if CACHE.exists():
    df = pd.read_csv(CACHE)
else:
    rows = []
    for split, ds in splits.items():
        for img, mask_path, sub in ds.items:
            m = decode_pothole_mix_mask(np.array(Image.open(mask_path).convert("RGB")))
            rows.append(dict(split=split, subset=sub, file=img.name, h=m.shape[0], w=m.shape[1],
                             crack=(m == CRACK).mean(), pothole=(m == POTHOLE).mean()))
    df = pd.DataFrame(rows)
    df.to_csv(CACHE, index=False)
df["damage"] = df.crack + df.pothole
df.head()

### 2a. Image resolutions per subset

In [ ]:
df["res"] = df.w.astype(str) + "x" + df.h.astype(str)
res = df.groupby("subset").res.agg(lambda r: ", ".join(f"{k} ({v})" for k, v in Counter(r).most_common(3)))
res.to_frame("most common resolutions (count)")

### 2b. Class content and imbalance per subset

In [ ]:
summary = df.groupby("subset").agg(
    images=("file", "size"),
    has_crack=("crack", lambda x: (x > 0).mean()),
    has_pothole=("pothole", lambda x: (x > 0).mean()),
    empty_mask=("damage", lambda x: (x == 0).mean()),
    damage_px_mean=("damage", "mean"),
    damage_px_median=("damage", "median"),
)
nrd_fg = np.array([(nrd[i][1] > 0).mean() for i in range(len(nrd))])
summary.loc["NRD (eval)"] = [len(nrd), np.nan, np.nan, (nrd_fg == 0).mean(), nrd_fg.mean(), np.median(nrd_fg)]
pct = [c for c in summary.columns if c != "images"]
summary[pct] = (100 * summary[pct]).round(2)
summary.rename(columns={c: c + " %" for c in pct})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
groups = [(s, g.damage[g.damage > 0]) for s, g in df.groupby("subset")] + [("NRD (eval)", nrd_fg[nrd_fg > 0])]
ax.boxplot([g for _, g in groups], tick_labels=[s for s, _ in groups], orientation="horizontal")
ax.set_xscale("log"); ax.set_xlabel("fraction of pixels that are damage (log, non-empty masks)")
ax.set_title("Damage pixel fraction per image")
plt.tight_layout(); plt.savefig(FIG / "damage_fraction.png", dpi=150); plt.show()

## 3. How wide are the cracks, and what happens when we resize?

Width estimate for thin structures: `width ≈ 2 · area / boundary_pixels` (a band of width w and length L has area wL and
~2L boundary pixels). Biased towards ~2 px for 1-px lines, which is fine for this purpose.
Then: the same crack after resizing the image to a given long side. **Below ~1.5 px the network can barely see it** →
train on crops at (near) native resolution instead of resizing whole images.

In [ ]:
def width_px(m):
    m = m.astype(bool)
    if not m.any():
        return np.nan
    p = np.pad(m, 1)
    inner = p[1:-1, 1:-1] & p[:-2, 1:-1] & p[2:, 1:-1] & p[1:-1, :-2] & p[1:-1, 2:]
    return 2 * m.sum() / max((m & ~inner).sum(), 1)

SIDES = [512, 768, 1024]
rows = []
train = splits["training"]
for sub, idx in pd.Series(range(len(train))).groupby([s for *_, s in train.items]):
    for i in rng.choice(idx.values, min(40, len(idx)), replace=False):
        m = train[i][1]
        if (m == CRACK).any():
            rows.append(dict(subset=sub, long_side=max(m.shape), width=width_px(m == CRACK)))
for i in range(len(nrd)):
    m = nrd[i][1]
    if m.any():
        rows.append(dict(subset="NRD (eval, all damage)", long_side=max(m.shape), width=width_px(m)))
w = pd.DataFrame(rows)
for L in SIDES:
    w[f"@{L}px"] = w.width * np.minimum(1, L / w.long_side)
crack_width = w.groupby("subset")[["long_side", "width"] + [f"@{L}px" for L in SIDES]].median().round(1)
crack_width.rename(columns={"long_side": "native long side", "width": "width native px"})

## 4. Visual comparison: training subsets vs NRD (domain gap)

In [ ]:
def overlay(img, m, alpha=0.5):
    out = img.astype(float).copy()
    out[m == CRACK] = (1 - alpha) * out[m == CRACK] + alpha * np.array([0, 255, 0])
    out[m == POTHOLE] = (1 - alpha) * out[m == POTHOLE] + alpha * np.array([255, 0, 0])
    return out.astype(np.uint8)

subsets = sorted(set(s for *_, s in train.items))
fig, axes = plt.subplots(3, len(subsets), figsize=(3.2 * len(subsets), 7.5))
for c, sub in enumerate(subsets):
    idx = [i for i, (*_, s) in enumerate(train.items) if s == sub]
    for r, i in enumerate(rng.choice(idx, 3, replace=False)):
        img, m = train[i]
        axes[r, c].imshow(overlay(img, m)); axes[r, c].axis("off")
    axes[0, c].set_title(sub, fontsize=9)
fig.suptitle("Pothole-Mix training subsets (green = crack, red = pothole)")
plt.tight_layout(); plt.savefig(FIG / "pothole_mix_samples.png", dpi=120); plt.show()

In [ ]:
order = np.argsort(-nrd_fg)  # most damage first, plus some empty ones at the end
pick = list(order[:6]) + list(np.where(nrd_fg == 0)[0][:2])
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5))
for ax, i in zip(axes.flat, pick):
    img, m = nrd[i]
    ax.imshow(overlay(img, m * CRACK)); ax.axis("off")
    ax.set_title(f"{nrd.items[i][0].name[:26]}  {nrd_fg[i]:.2%}", fontsize=8)
fig.suptitle("Norwegian Road Damage (eval) - damage mask in green; last two have no annotations")
plt.tight_layout(); plt.savefig(FIG / "nrd_samples.png", dpi=120); plt.show()

### 4b. Annotation tightness in NRD

Zoom in on the largest annotated regions: do the polygons trace the cracks, or are they loose outlines around damaged areas?
This matters for how to interpret IoU on NRD.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, i in zip(axes, order[:4]):
    img, m = nrd[i]
    ys, xs = np.where(m)
    y0, y1, x0, x1 = max(ys.min() - 20, 0), ys.max() + 20, max(xs.min() - 20, 0), xs.max() + 20
    ax.imshow(img[y0:y1, x0:x1]); ax.contour(m[y0:y1, x0:x1], levels=[0.5], colors="lime", linewidths=1)
    ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "nrd_annotation_zoom.png", dpi=150); plt.show()

## 5. NRD annotation categories

In [ ]:
cats = Counter()
for split in ("train", "test"):
    coco = json.loads((NRD / split / "_annotations.coco.json").read_text())
    names = {c["id"]: c["name"] for c in coco["categories"]}
    cats.update(names[a["category_id"]] for a in coco["annotations"])
pd.Series(cats, name="annotations").sort_values(ascending=False).to_frame()

## 6. Findings → decisions

*First pass, written from the outputs on 2026-11-18. Re-check if the data or code changes.*

- **Two annotation styles, not one.** Pothole-Mix traces cracks tightly as thin lines (and potholes as filled regions).
  NRD outlines whole *damaged regions* with loose polygons (bands around crack zones, patches, ravelling).
  A zero-shot model trained on Pothole-Mix will predict thin crack lines inside NRD's polygons, so **pixel IoU on NRD is
  capped by label style, not only by model quality.** Report it, but also report a looser measure (e.g. region/object-level
  hit rate, or IoU after dilating predictions) and discuss it. This is also the strongest argument for the later
  leave-one-video-out NRD fine-tune: it adapts the *label style*, not just the domain.
- **Resolution:** crack width in edmcrack600 / gaps384 / cracks-and-potholes drops to ~1.5-1.7 px when the long side is
  512, i.e. at the limit of visibility. → **train on random crops (e.g. 512x512) at native or ~1024 long-side resolution**
  instead of resizing whole images to 512. NRD regions are wide (~16 px native) so evaluation at 1280x720 is fine.
- **Imbalance:** median damage fraction is 0.3-3% for crack subsets and 0.5% on NRD; 10% of NRD images are empty.
  → Dice + BCE (or focal) loss, not plain BCE; report precision and recall, not just IoU.
- **Sampling:** cracks-and-potholes-in-road is 60% of training images; cnr-road-dataset is 17 images.
  → consider subset-balanced sampling as an experiment.
- **Domain gap:** crack500, gaps384 and pothole600 are top-down close-ups; cracks-and-potholes-in-road and edmcrack600 are
  forward-facing road views closest to NRD's dashcam frames (NRD also shows the car bonnet, sky, snow, guard rails).
  → natural experiment: all of Pothole-Mix vs only the dashcam-like subsets, evaluated on NRD.
- **NRD labels:** 84 of 109 annotations are the generic class; D10 absent; only 4 potholes → binary is the right choice.